In [5]:
from src.authorization.auth import *
import geemap
import ee
from datetime import datetime, timezone

In [6]:
def get_image(coordinates: list[float],
                   image_collection: str,
                   start_date: str,
                   end_date: str,
                   cloud_percentage: float | int
                   ) -> tuple[str, ee.Geometry]:

    roi = ee.Geometry.Point(coordinates).buffer(450).bounds()

    image = (
        ee.ImageCollection(image_collection)
        .filterBounds(roi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', cloud_percentage))
        .sort('CLOUDY_PIXEL_PERCENTAGE')
        .first()
    )
    full_id = image.get('system:id').getInfo()

    return full_id, roi


def get_metadata(image_id: str) -> dict[str, str]:
    img = ee.Image(image_id)
    props = img.toDictionary().getInfo()

    time_start = props.get('system:time_start')
    acquired_at = (
        datetime.fromtimestamp(time_start/1000, tz=timezone.utc).isoformat()
        if time_start else None
    )

    return {
        "system_id": image_id,
        "acquired_at": acquired_at,
        "cloud_pct": props.get("CLOUDY_PIXEL_PERCENTAGE"),
        "mgrs_tile": props.get("MGRS_TILE"),
        "product_id": props.get("PRODUCT_ID"),
        "platform": props.get("SPACECRAFT_NAME"),
        "processing_baseline": props.get("PROCESSING_BASELINE"),
    }

In [7]:
credentials = authenticate_google_api()
initialize_earth_engine(credentials)


data_to_find = {
    "roi_coordinates":[22.229681, 50.554120],
    "collection":'COPERNICUS/S2_SR_HARMONIZED',
    "start_date":'2024-09-01',
    "end_date":'2024-10-31',
    "cloud_percentage": 20
}
img_id, roi = get_image(coordinates=data_to_find.get('roi_coordinates'),
                        image_collection=data_to_find.get("collection"),
                        start_date=data_to_find.get("start_date"),
                        end_date=data_to_find.get("end_date"),
                        cloud_percentage=data_to_find.get("cloud_percentage"),
                        )
img_metadata = get_metadata(img_id)
print(img_metadata)

Earth Engine initialized successfully!
{'system_id': 'COPERNICUS/S2_SR_HARMONIZED/20240920T094029_20240920T094449_T34UEB', 'acquired_at': None, 'cloud_pct': 0.002273, 'mgrs_tile': '34UEB', 'product_id': 'S2B_MSIL2A_20240920T094029_N0511_R036_T34UEB_20240920T122840', 'platform': 'Sentinel-2B', 'processing_baseline': '05.11'}


In [8]:
img_metadata

{'system_id': 'COPERNICUS/S2_SR_HARMONIZED/20240920T094029_20240920T094449_T34UEB',
 'acquired_at': None,
 'cloud_pct': 0.002273,
 'mgrs_tile': '34UEB',
 'product_id': 'S2B_MSIL2A_20240920T094029_N0511_R036_T34UEB_20240920T122840',
 'platform': 'Sentinel-2B',
 'processing_baseline': '05.11'}